In [1]:
import numpy as np
import pandas as pd

In [5]:
# Load the dataset
data = pd.read_csv("Vahan_Case_Study.csv")
data.head()

,upload_date,lead_source,candidate_phone,Uploaded Leads,Attempted,Connected,Attempt per Lead,tag_filled,Interested,OB_after_upload,OB_after_first_attempt,FT_after_upload,FT_after_first_attempt,upload_to_first_attempt_P50 (hrs),Attempted %,Attempt to Connected %,Connect to Interested %,Interested to FT_after_first_attempt %,Attempted to FT_after_upload %
0,18-07-2026,2W3W - 3WAHD - Khanna - 3W - 17 Jul,91148e89d497914604c9c248ccd2aca212fac665bdc396...,1,1,1,1.0,1,0,0,0,0,0,112.0,100,100.0,0.0,NaN,0.0
1,18-07-2026,2W3W - 3WAHD - Khanna - 3W - 17 Jul,4aa4e7a4f4a3b9fa608c4c1729be6a8accc31f58b3d8b1...,1,1,0,1.0,0,0,0,0,0,0,112.0,100,0.0,NaN,NaN,0.0
2,18-07-2026,2W3W - 3WAHD - Khanna - 3W - 17 Jul,0e97e34090259c1ce6cef1fd1c491fcfc469ba9d2d1ce7...,1,1,0,1.0,0,0,0,0,0,0,89.0,100,0.0,NaN,NaN,0.0
3,18-07-2026,2W3W - 3WAHD - Khanna - 3W - 17 Jul,cdc003718802cffa769a478bcc0fe31aa20ac3ffe97a7c...,1,1,0,1.0,0,0,0,0,0,0,106.0,100,0.0,NaN,NaN,0.0
4,18-07-2026,2W3W - 3WAHD - Khanna - 3W - 17 Jul,d70833a770ab63e1992254dad148b100bbbadbe0e94673...,1,1,0,1.0,0,0,0,0,0,0,90.0,100,0.0,NaN,NaN,0.0


## Cohort analysis: which `lead_source` cohorts convert best?

North-star metric: `FT_after_upload` rate (leads that make it all the way to Field Training), not intermediate
funnel steps like connect/interest rate. Cohorts with too few leads are excluded, and "warm" cohorts
(already pre-qualified, e.g. fees already paid) are ranked separately from "cold" acquisition cohorts since
they aren't playing the same game and shouldn't be pooled into one ranking.

In [6]:
cohort = data.groupby('lead_source').agg(
    leads=('Uploaded Leads', 'sum'),
    attempted=('Attempted', 'sum'),
    connected=('Connected', 'sum'),
    interested=('Interested', 'sum'),
    ft_after_upload=('FT_after_upload', 'sum'),
    ft_after_first_attempt=('FT_after_first_attempt', 'sum'),
).reset_index()

cohort['attempt_rate']  = cohort['attempted'] / cohort['leads'] * 100
cohort['connect_rate']  = cohort['connected'] / cohort['attempted'] * 100
cohort['interest_rate'] = cohort['interested'] / cohort['connected'] * 100
cohort['ft_rate']       = cohort['ft_after_upload'] / cohort['leads'] * 100

cohort.sort_values('ft_rate', ascending=False)

,lead_source,leads,attempted,connected,interested,ft_after_upload,ft_after_first_attempt,attempt_rate,connect_rate,interest_rate,ft_rate
15,Single Referral > 7 days- 24th Jul,1500,1454,690,7,14,2,96.933333,47.455296,1.014493,0.933333
11,Khanna- 2W 26th Jul,1546,1376,574,20,14,6,89.003881,41.715116,3.484321,0.905563
13,PreOb-Ob Fees Paid 29th Jul (set 1),1483,1433,650,103,7,2,96.628456,45.359386,15.846154,0.472016
14,PreOb-Ob Fees Paid 29th Jul (set 2),1558,1519,735,122,7,5,97.496791,48.387097,16.598639,0.449294
6,AI Connected but not Connected by TC- Set 1,1480,1331,694,29,5,0,89.932432,52.141247,4.178674,0.337838
7,AI Connected but not Connected by TC- Set 2,1193,1066,466,26,2,0,89.354568,43.714822,5.579399,0.167645
10,Khanna AI,886,245,137,2,1,0,27.652370,55.918367,1.459854,0.112867
12,OLX - Ashwin - 2W - 17 Jul,5182,962,372,9,4,2,18.564261,38.669439,2.419355,0.077190
4,50K 2W5 - Khanna - 2W - 17 Jul,1,0,0,0,0,0,0.000000,NaN,NaN,0.000000
5,AI Connected band Not Interested,2137,1359,637,21,0,0,63.593823,46.872701,3.296703,0.000000


In [7]:
MIN_LEADS = 100  # drop cohorts too small to draw a conclusion from (e.g. n=1, n=2)

# "Warm" cohorts are already pre-qualified (fees paid / AI pre-screened) rather than
# cold acquisition traffic, so they aren't comparable to the rest on funnel-rate metrics.
warm_keywords = ['PreOb', 'AI Connected']
cohort['segment'] = cohort['lead_source'].apply(
    lambda s: 'warm' if any(k in s for k in warm_keywords) else 'cold'
)

ranked = cohort[cohort['leads'] >= MIN_LEADS].sort_values('ft_rate', ascending=False)
ranked[['lead_source', 'segment', 'leads', 'ft_after_upload', 'ft_rate', 'connect_rate', 'interest_rate']]

,lead_source,segment,leads,ft_after_upload,ft_rate,connect_rate,interest_rate
15,Single Referral > 7 days- 24th Jul,cold,1500,14,0.933333,47.455296,1.014493
11,Khanna- 2W 26th Jul,cold,1546,14,0.905563,41.715116,3.484321
13,PreOb-Ob Fees Paid 29th Jul (set 1),warm,1483,7,0.472016,45.359386,15.846154
14,PreOb-Ob Fees Paid 29th Jul (set 2),warm,1558,7,0.449294,48.387097,16.598639
6,AI Connected but not Connected by TC- Set 1,warm,1480,5,0.337838,52.141247,4.178674
7,AI Connected but not Connected by TC- Set 2,warm,1193,2,0.167645,43.714822,5.579399
10,Khanna AI,cold,886,1,0.112867,55.918367,1.459854
12,OLX - Ashwin - 2W - 17 Jul,cold,5182,4,0.077190,38.669439,2.419355
5,AI Connected band Not Interested,warm,2137,0,0.000000,46.872701,3.296703
2,2W3W - 3WEV - Khanna - 3W - 17 Jul,cold,201,0,0.000000,45.273632,1.098901


In [ ]:
cols = ['lead_source', 'leads', 'ft_after_upload', 'ft_rate']

print("Top 3 cold-lead cohorts by FT rate (leads -> FT_after_upload):")
display(ranked[ranked['segment'] == 'cold'].head(3)[cols])

print("\nTop 3 warm-lead cohorts by FT rate:")
display(ranked[ranked['segment'] == 'warm'].head(3)[cols])

Top 3 cold-lead cohorts by FT rate (leads -> FT_after_upload):


,lead_source,leads,ft_after_upload,ft_rate
15,Single Referral > 7 days- 24th Jul,1500,14,0.933333
11,Khanna- 2W 26th Jul,1546,14,0.905563
10,Khanna AI,886,1,0.112867



Top 3 warm-lead cohorts by FT rate:


,lead_source,leads,ft_after_upload,ft_rate
13,PreOb-Ob Fees Paid 29th Jul (set 1),1483,7,0.472016
14,PreOb-Ob Fees Paid 29th Jul (set 2),1558,7,0.449294
6,AI Connected but not Connected by TC- Set 1,1480,5,0.337838


: 